In [ ]:
from slicer.ScriptedLoadableModule import *
from __main__ import vtk, qt, ctk, slicer
from vtk.util import numpy_support
import sitkUtils as su
import os

def mri_registration(fixed_image, mobile_image, transformation):
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(fixed_image)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0)
    resampler.SetTransform(transformation)
    image_registration = resampler.Execute(mobile_image)
    return image_registration

def multires_registrations(fixed_image, moving_image, initial_transform, ImageSamplingPercentage):
    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins = 50)
    registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
    registration_method.SetMetricSamplingPercentage(float(ImageSamplingPercentage)/100)
    registration_method.SetInterpolator(sitk.sitkLinear)
    registration_method.SetOptimizerAsGradientDescent(learningRate = 1.0, estimateLearningRate = registration_method.EachIteration, numberOfIterations = 100)
    registration_method.SetOptimizerScalesFromPhysicalShift()
    registration_method.SetInitialTransform(initial_transform)
    registration_method.SetShrinkFactorsPerLevel(shrinkFactors = [4, 2, 1])
    registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas = [3, 2, 1])
    registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    final_transform = registration_method.Execute(fixed_image, moving_image)
    print('Final metric value: {0}'.format(registration_method.GetMetricValue()))
    print('Optimizer\'s stopping condition, {0}'.format(registration_method.GetOptimizerStopConditionDescription()))
    
    return final_transform

def main_image_Traitement_MRI(fixed_mri, moving_mri, percentage):
    moving_mri = sitk.Cast(moving_mri, sitk.sitkFloat32)
    fixed_mri     = sitk.Cast(fixed_mri, sitk.sitkFloat32)
    moving_mri.SetDirection(fixed_mri.GetDirection())
    moving_mri.SetOrigin(fixed_mri.GetOrigin())
    
    initial_transform = sitk.CenteredTransformInitializer(fixed_mri, moving_mri, sitk.Euler3DTransform(), sitk.CenteredTransformInitializerFilter.MOMENTS)
    
    medium_transform = multires_registrations(fixed_mri, moving_mri, initial_transform, percentage)
    
    moving_mri = mri_registration(fixed_mri, moving_mri, medium_transform)
    su.PushVolumeToSlicer(moving_mri, name = 'MRIregistertoCT', className = 'vtkMRMLScalarVolumeNode')

    return moving_mri

In [ ]:
import SimpleITK as sitk

input_data_directory   = "XXX"
output_directory   = "XXX"
images_reg = {}
files = os.listdir(input_data_directory)

t0_dict = {
        f.split('_')[0]: f
        for f in files
        if 't0_t1gd' in f
    }

for files in files:
    if 't0' in files:
        parts = files.split('_')
        ref_name = parts[0]
        if ref_name not in t0_dict:
            print(f"Error")
            continue

        input_path = os.path.join(input_data_directory, t0_dict[ref_name])
        output_path = os.path.join(input_data_directory, files)

        image_pre = sitk.ReadImage(input_path)
        image_post = sitk.ReadImage(output_path)

        image_post_reg = main_image_Traitement_MRI(image_pre, image_post, 20)

        output_filename = f"{files}_w.nii"
        output_path = os.path.join(output_directory, output_filename)

        sitk.WriteImage(image_post_reg, output_path)

for files in files:
    if 't1' in files:
        parts = files.split('_')
        ref_name = parts[0]
        if ref_name not in t0_dict:
            print(f"Error")
            continue

        input_path = os.path.join(input_data_directory, t0_dict[ref_name])
        output_path = os.path.join(input_data_directory, files)

        image_pre = sitk.ReadImage(input_path)
        image_post = sitk.ReadImage(output_path)

        image_post_reg = main_image_Traitement_MRI(image_pre, image_post, 20)

        output_filename = f"{files}_w.nii"
        output_path = os.path.join(output_directory, output_filename)

        sitk.WriteImage(image_post_reg, output_path)

for files in files:
    if 't2' in files:
        parts = files.split('_')
        ref_name = parts[0]
        if ref_name not in t0_dict:
            print(f"Error")
            continue

        input_path = os.path.join(input_data_directory, t0_dict[ref_name])
        output_path = os.path.join(input_data_directory, files)

        image_pre = sitk.ReadImage(input_path)
        image_post = sitk.ReadImage(output_path)

        image_post_reg = main_image_Traitement_MRI(image_pre, image_post, 20)

        output_filename = f"{files}_w.nii"
        output_path = os.path.join(output_directory, output_filename)

        sitk.WriteImage(image_post_reg, output_path)